In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
# Crear widgets
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsfinalproject")

In [0]:
# Obtener valores de widgets
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

# Obtener ruta de archivo constructors.json
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/constructors.json"

In [0]:
# Definir esquema para el Dataframe de constructors
constructors_schema = StructType(fields=[
    StructField("constructorId", IntegerType(), False),
    StructField("constructorRef", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [0]:
# Crear DataFrame df_constructors con el esquema y ruta definidos.
df_constructors = spark.read \
    .schema(constructors_schema) \
    .option("multiLine", True) \
    .json(ruta)

In [0]:
# Cambiar nombres de columnas
constructors_final_df = df_constructors.select(
    col("constructorId").alias("constructor_id"),
    col("constructorRef").alias("constructor_ref"),
    col("name"),
    col("nationality")
).withColumn("ingestion_date", current_timestamp())

In [0]:
# Ingestar data en la tabla constructors.
constructors_final_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.constructors")